<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/VARSR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture
import os
import shutil
import re

!nvidia-smi

%cd /content
if not os.path.exists("/content/VARSR"):
    !git clone https://github.com/quyp2000/VARSR.git

%cd /content/VARSR

# Clean requirements.txt from the non-existent mc==0.0 package and install dependencies
!sed -i '/mc==0.0/d' requirements.txt
!pip install -q -r requirements.txt
!pip install -q typed-argument-parser pyiqa einops timm huggingface_hub gdown

# Create the working folder structure
os.makedirs("/content/VARSR/checkpoints", exist_ok=True)
os.makedirs("/content/VARSR/upload", exist_ok=True)
os.makedirs("/content/VARSR/results", exist_ok=True)
os.makedirs("/content/VARSR/testset/custom/LR", exist_ok=True)

# Download weights directly from Hugging Face (if not already downloaded)
from huggingface_hub import hf_hub_download

checkpoints_dir = "/content/VARSR/checkpoints"
for weight_file in ["VQVAE.pth", "VARSR.pth"]:
    target = os.path.join(checkpoints_dir, weight_file)
    if not os.path.exists(target):
        print(f"⏳ Downloading {weight_file} from Hugging Face...")
        hf_hub_download(
            repo_id="qyp2000/VARSR",
            filename=weight_file,
            local_dir=checkpoints_dir,
            local_dir_use_symlinks=False
        )

# Patch test_tile.py: inject mmap=True (protects against OOM) and disable the crash in metrics()
with open("test_tile.py", "r") as f:
    code = f.read()

# 1. mmap for VQVAE
code = code.replace(
    "vae.load_state_dict(torch.load(vae_ckpt, map_location='cpu')['trainer']['vae_local'], strict=True)",
    "vae_dict = torch.load(vae_ckpt, map_location='cpu', mmap=True)\n    vae.load_state_dict(vae_dict['trainer']['vae_local'] if 'trainer' in vae_dict else vae_dict, strict=False)\n    del vae_dict"
)

# 2. mmap for VARSR (prevents exceeding Colab's 12.7 GB memory limit)
code = code.replace(
    "model_state = torch.load(var_ckpt, map_location='cpu')",
    "model_state = torch.load(var_ckpt, map_location='cpu', mmap=True)"
)
code = code.replace(
    "var.load_state_dict(model_state['trainer']['var_wo_ddp'], strict=True)",
    "var.load_state_dict(model_state['trainer']['var_wo_ddp'] if 'trainer' in model_state else model_state, strict=True)\n    del model_state\n    import gc; gc.collect()"
)

# 3. Disable the call to the broken metrics() function
code = re.sub(r'results\s*=\s*metrics\(\)', '# results = metrics()', code)

with open("test_tile.py", "w") as f:
    f.write(code)

print("\n✅ Environment ready, weights verified, script optimized for GPU!")

In [ ]:
#@title ##**Upload images** { display-mode: "form" }
import os
import shutil
import glob
import PIL.Image
from google.colab import files

%cd /content/VARSR

upload_dir = "/content/VARSR/upload"
target_lr = "/content/VARSR/testset/custom/LR"

# Clear folders before a new upload
for folder in [upload_dir, target_lr]:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

# Select files from your computer
uploaded = files.upload()

valid_extensions = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff')
processed_count = 0

for filename in uploaded.keys():
    if filename.lower().endswith(valid_extensions):
        src_path = os.path.join(os.getcwd(), filename)
        shutil.copy(src_path, os.path.join(upload_dir, filename))

        # The model strictly requires .png format: convert on the fly
        base_name = os.path.splitext(filename)[0]
        dst_png = os.path.join(target_lr, f"{base_name}.png")

        img = PIL.Image.open(src_path).convert("RGB")
        img.save(dst_png, "PNG")
        processed_count += 1

print(f"\n✅ Successfully uploaded and prepared for processing: {processed_count} file(s)")

In [ ]:
#@title ##**Run** { display-mode: "form" }
import os
import shutil
import glob
import re
import subprocess
import math
import PIL.Image
from tqdm.notebook import tqdm

%cd /content/VARSR

# Protection against GPU memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

#@markdown ### ⚙️ **Scaling parameters:**
scale = 4.0 #@param {type:"number"}
cfg_guidance = 7.0 #@param {type:"slider", min:1.0, max:12.0, step:0.5}

upload_dir = "/content/VARSR/upload"
target_lr = "/content/VARSR/testset/custom/LR"
results_dir = "/content/VARSR/results"
prediction_dir = "/content/VARSR/testset/custom/VARPrediction"

# Wipe every folder from the previous run so old output photos never survive a new upload
for folder in [target_lr, results_dir, prediction_dir]:
    if os.path.isdir(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

# 1. Prepare and convert images to PNG

valid_exts = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.tiff')
user_images = [f for f in os.listdir(upload_dir) if f.lower().endswith(valid_exts)]
if len(user_images) == 0:
    raise FileNotFoundError("❌ No images found in the upload folder! Upload them first using Cell 2.")

for img_name in user_images:
    src_p = os.path.join(upload_dir, img_name)
    base_n = os.path.splitext(img_name)[0]
    dst_p = os.path.join(target_lr, f"{base_n}.png")
    img = PIL.Image.open(src_p).convert("RGB")
    img.save(dst_p, "PNG")

# 2. Reset and inject the patch with progress-bar hooks
!git checkout test_tile.py > /dev/null 2>&1

with open("test_tile.py", "r") as f:
    lines = f.readlines()

new_lines = []
i = 0
while i < len(lines):
    line = lines[i]

    # VQVAE mmap patch
    if "vae.load_state_dict(torch.load(vae_ckpt, map_location='cpu')" in line:
        sp = " " * (len(line) - len(line.lstrip()))
        new_lines.append(f"{sp}vae_state = torch.load(vae_ckpt, map_location='cpu', mmap=True)\n")
        new_lines.append(f"{sp}vae.load_state_dict(vae_state['trainer']['vae_local'] if 'trainer' in vae_state else vae_state, strict=False)\n")
        new_lines.append(f"{sp}del vae_state\n")
        i += 1
        continue

    # VARSR mmap patch
    if "model_state = torch.load(var_ckpt, map_location='cpu')" in line:
        sp = " " * (len(line) - len(line.lstrip()))
        new_lines.append(f"{sp}model_state = torch.load(var_ckpt, map_location='cpu', mmap=True)\n")
        i += 1
        continue

    # var_wo_ddp patch and RAM cleanup
    if "var.load_state_dict(model_state['trainer']['var_wo_ddp'], strict=True)" in line:
        sp = " " * (len(line) - len(line.lstrip()))
        new_lines.append(f"{sp}var.load_state_dict(model_state['trainer']['var_wo_ddp'] if 'trainer' in model_state else model_state, strict=True)\n")
        new_lines.append(f"{sp}del model_state\n")
        new_lines.append(f"{sp}import gc; gc.collect()\n")
        i += 1
        continue

    # Disable the call to the broken metrics
    if "results = metrics()" in line:
        sp = " " * (len(line) - len(line.lstrip()))
        new_lines.append(f"{sp}# results = metrics()\n")
        i += 1
        continue

    # rscale scale factor
    if "rscale = 4" in line:
        sp = " " * (len(line) - len(line.lstrip()))
        new_lines.append(f"{sp}rscale = {int(scale)}\n")
        i += 1
        continue

    # Hook before the image loop: expose the filename to the outer process
    if "for image_name in image_names[:]:" in line:
        sp = " " * (len(line) - len(line.lstrip()))
        new_lines.append(f"{sp}for image_name in image_names[:]:\n")
        new_lines.append(f"{sp}    print(f'@@IMG@@:{{os.path.basename(image_name)}}', flush=True)\n")
        i += 1
        continue

    # Adaptive tiling with a progress hook for each tile
    if "recon_B3HW = var.autoregressive_infer_cfg(" in line:
        sp = " " * (len(line) - len(line.lstrip()))
        while i < len(lines) and "more_smooth=False)" not in lines[i]:
            i += 1
        i += 1

        new_lines.append(f"{sp}_chunk_size = 1\n")
        new_lines.append(f"{sp}_recon_chunks = []\n")
        new_lines.append(f"{sp}for _ci in range(0, grid_cols, _chunk_size):\n")
        new_lines.append(f"{sp}    _sub_lr = lr4var[_ci:_ci + _chunk_size]\n")
        new_lines.append(f"{sp}    _sub_B = _sub_lr.shape[0]\n")
        new_lines.append(f"{sp}    _sub_recon = var.autoregressive_infer_cfg(B=_sub_B, cfg={cfg_guidance}, top_k=1, top_p=0.75, text_hidden=None, lr_inp=_sub_lr, negative_text=None, label_B=label_B.repeat(_sub_B), lr_inp_scale=None, tile_flag=True, more_smooth=False)\n")
        new_lines.append(f"{sp}    _recon_chunks.append(_sub_recon)\n")
        new_lines.append(f"{sp}    torch.cuda.empty_cache()\n")
        new_lines.append(f"{sp}    print('@@TICK@@', flush=True)\n")
        new_lines.append(f"{sp}recon_B3HW = torch.cat(_recon_chunks, dim=0)\n")
        continue

    new_lines.append(line)
    i += 1

with open("test_tile.py", "w") as f:
    f.writelines(new_lines)

# 3. Precisely calculate the total number of tiles for the progress bar
total_tiles = 0
for img_name in user_images:
    p_img = PIL.Image.open(os.path.join(upload_dir, img_name))
    w_px, h_px = p_img.size
    tw = max(math.ceil(w_px / 16) * 16 * int(scale), 512)
    th = max(math.ceil(h_px / 16) * 16 * int(scale), 512)
    g_rows = math.ceil(((th // 16) - 32) / 24) + 1
    g_cols = math.ceil(((tw // 16) - 32) / 24) + 1
    total_tiles += g_rows * g_cols

# 4. Run the process in the background with log filtering
proc = subprocess.Popen(
    ["python", "test_tile.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

captured_logs = []
pbar = tqdm(total=total_tiles, desc="🚀 4K Upscale", unit="tile")

for raw_line in iter(proc.stdout.readline, ""):
    line = raw_line.strip()
    if not line:
        continue

    if line.startswith("@@IMG@@:"):
        current_img = line.replace("@@IMG@@:", "")
        pbar.set_description(f"🖼️ {current_img}")
    elif line == "@@TICK@@":
        pbar.update(1)
    else:
        captured_logs.append(line)

proc.stdout.close()
proc.wait()
pbar.close()

# 5. Collect results
generated_files = glob.glob(f"{prediction_dir}/*.png")
for file_path in generated_files:
    shutil.copy(file_path, os.path.join(results_dir, os.path.basename(file_path)))

if proc.returncode == 0 and len(generated_files) > 0:
    print(f"\n🎉 ALL DONE! Files processed: {len(generated_files)}")
    print(f"📁 Results are located in: {results_dir}")
    print("Run Cell 4 (Visualize) to see the Before/After comparison!")
else:
    print("\n❌ An error occurred during execution. Logs:")
    print("\n".join(captured_logs[-15:]))

In [ ]:
#@title ##**4. Visualize** { display-mode: "form" }
import os
import PIL.Image
import numpy as np
from IPython.display import HTML, display
import base64
from io import BytesIO

upload_folder = "/content/VARSR/upload"
result_folder = "/content/VARSR/results"

visualization_method = "Slider" #@param ["Side-by-Side", "Slider"]

# Support different Pillow versions for the Lanczos filter
resample_filter = getattr(getattr(PIL.Image, 'Resampling', PIL.Image), 'LANCZOS', PIL.Image.LANCZOS)

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), resample_filter)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), resample_filter)
    return image

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

# Dynamic lookup, no hardcoding
filenames_upload = sorted([f for f in os.listdir(upload_folder) if is_image_file(f)]) if os.path.exists(upload_folder) else []
filenames_result = sorted([f for f in os.listdir(result_folder) if is_image_file(f)]) if os.path.exists(result_folder) else []

# Match files by their base name (regardless of jpg/png extension)
paired_images = []
for f_up in filenames_upload:
    base_stem = os.path.splitext(f_up)[0]
    matched_res = next((f_res for f_res in filenames_result if os.path.splitext(f_res)[0] == base_stem), None)
    if matched_res:
        paired_images.append((f_up, matched_res))

if not paired_images:
    print(f"⚠️ No matching images found in {upload_folder} or {result_folder}.")
else:
    for idx, (filename, filename_output) in enumerate(paired_images):
        try:
            image_original = PIL.Image.open(os.path.join(upload_folder, filename)).convert('RGB')
            image_restore = PIL.Image.open(os.path.join(result_folder, filename_output)).convert('RGB')

            w_orig, h_orig = image_original.size
            w_res, h_res = image_restore.size

            print(f"[{filename}] Original: {w_orig}x{h_orig} px  ➔  VARSR 4K: {w_res}x{h_res} px")

            if visualization_method == "Side-by-Side":
                max_width = 500
                image_original = resize_image_maintain_aspect(image_original, max_width)
                image_restore = resize_image_maintain_aspect(image_restore, max_width)
                target_height = min(image_original.size[1], image_restore.size[1])
                image_original = resize_image_maintain_aspect(image_original, max_width, target_height)
                image_restore = resize_image_maintain_aspect(image_restore, max_width, target_height)

                combined_width = image_original.size[0] + image_restore.size[0]
                combined_image = PIL.Image.new('RGB', (combined_width, target_height))
                combined_image.paste(image_original, (0, 0))
                combined_image.paste(image_restore, (image_original.size[0], 0))
                display(combined_image)
                print("-" * 65)

            else:
                max_width = min(image_restore.size[0], 900)
                image_restore = resize_image_maintain_aspect(image_restore, max_width)
                target_height = image_restore.size[1]
                image_original = resize_image_maintain_aspect(image_original, max_width, target_height)

                original_base64 = image_to_base64(image_original)
                restore_base64 = image_to_base64(image_restore)

                unique_id = f"slider_container_{idx}"

                html_code = f"""
                <div id="{unique_id}" style="position: relative; width: {image_restore.size[0]}px; height: {image_restore.size[1]}px; margin: 15px 0 35px 0; user-select: none;">
                    <div style="position: relative; width: 100%; height: 100%; overflow: hidden; border-radius: 6px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
                        <img src="data:image/png;base64,{original_base64}" style="position: absolute; top:0; left:0; width: 100%; height: 100%; pointer-events: none;">
                        <div class="clip-target" style="position: absolute; top:0; left:0; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%); pointer-events: none;">
                            <img src="data:image/png;base64,{restore_base64}" style="position: absolute; top:0; left:0; width: 100%; height: 100%; pointer-events: none;">
                        </div>
                    </div>
                    <div class="slider-bar" style="position: absolute; top: 0; bottom: 0; width: 4px; background: #ffffff; cursor: ew-resize; left: 50%; box-shadow: 0 0 8px rgba(0,0,0,0.6);">
                        <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 26px; height: 26px; background: #ffffff; border-radius: 50%; left: -11px; box-shadow: 0 2px 6px rgba(0,0,0,0.4); display: flex; align-items: center; justify-content: center;">
                            <span style="color: #444; font-size: 11px; font-weight: bold; pointer-events: none;">◀▶</span>
                        </div>
                    </div>
                </div>
                <script>
                    (function() {{
                        const root = document.getElementById('{unique_id}');
                        if (!root) return;
                        const slider = root.querySelector('.slider-bar');
                        const clipDiv = root.querySelector('.clip-target');
                        let isDragging = false;

                        function updatePosition(clientX) {{
                            const rect = root.getBoundingClientRect();
                            let x = clientX - rect.left;
                            if (x < 0) x = 0;
                            if (x > rect.width) x = rect.width;
                            const percentage = (x / rect.width) * 100;
                            slider.style.left = percentage + '%';
                            clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                        }}

                        slider.addEventListener('mousedown', (e) => {{
                            isDragging = true;
                            e.preventDefault();
                        }});

                        window.addEventListener('mouseup', () => {{ isDragging = false; }});
                        window.addEventListener('mousemove', (e) => {{
                            if (!isDragging) return;
                            updatePosition(e.clientX);
                        }});

                        slider.addEventListener('touchstart', (e) => {{
                            isDragging = true;
                        }}, {{passive: true}});

                        window.addEventListener('touchend', () => {{ isDragging = false; }});
                        window.addEventListener('touchmove', (e) => {{
                            if (!isDragging || !e.touches[0]) return;
                            updatePosition(e.touches[0].clientX);
                        }}, {{passive: true}});
                    }})();
                </script>
                """
                display(HTML(html_code))

        except Exception as e:
            print(f"Error processing {filename}: {e}")

In [ ]:
#@title ##**Download results** { display-mode: "form" }
import os
from google.colab import files

results_dir = "/content/VARSR/results"
zip_name = "VARSR_results.zip"
zip_path = f"/content/VARSR/{zip_name}"

files_in_folder = [f for f in os.listdir(results_dir) if os.path.isfile(os.path.join(results_dir, f))]

if len(files_in_folder) == 0:
    print("❌ The results folder is empty.")
elif len(files_in_folder) == 1:
    # If there is only one file, download it directly without archiving
    files.download(os.path.join(results_dir, files_in_folder[0]))
else:
    # If there are several, pack them into a zip
    if os.path.exists(zip_path):
        os.remove(zip_path)
    !cd {results_dir} && zip -r -j {zip_path} *
    files.download(zip_path)